# 01 — Exploration du corpus et des questions

**Projet** : Assistant documentaire RAG avec scikit-learn (BM25) — baseline lexicale de référence

## Objectifs pédagogiques

1. Décrire un corpus documentaire avant de l'indexer : sources, sections, longueurs, dates de publication.
1. Lire un jeu de questions annotées : difficultés, intentions, questions hors corpus et nombre de documents pertinents par question.
1. Comprendre pourquoi la part de questions hors corpus conditionne la lecture de toutes les métriques de rappel.

**Objectifs transverses du dépôt**

- Découper un corpus en passages : pourquoi le chevauchement existe et comment mesurer son effet sur le rappel.
- Comprendre BM25 (saturation, normalisation de longueur, IDF) en l'implémentant, plutôt qu'en l'appelant.
- Évaluer un système de recherche par question puis moyenne : recall@k, MRR, nDCG@k et ce que chacun ignore.

In [ ]:
import sys
from pathlib import Path

import matplotlib.pyplot as plt
import pandas as pd
from IPython.display import display

# --- Racine du projet ---------------------------------------------------------------------------
# Le notebook s'exécute depuis `notebooks/` : on remonte d'un cran pour pouvoir importer `src`.
PROJECT_ROOT = Path.cwd().resolve()
if PROJECT_ROOT.name == "notebooks":
    PROJECT_ROOT = PROJECT_ROOT.parent
if str(PROJECT_ROOT) not in sys.path:
    sys.path.insert(0, str(PROJECT_ROOT))

from hydra import compose, initialize_config_dir  # noqa: E402
from hydra.core.global_hydra import GlobalHydra  # noqa: E402

from src.schemas.config import validate_config  # noqa: E402
from src.utils.config_access import node  # noqa: E402
from src.utils.logging import setup_logging  # noqa: E402
from src.utils.paths import ProjectPaths  # noqa: E402

# --- Réglages d'affichage -----------------------------------------------------------------------
plt.rcParams.update({"figure.dpi": 110, "axes.grid": True, "grid.alpha": 0.25})
pd.set_option("display.max_columns", 40)
pd.set_option("display.width", 170)
# Le projet journalise au niveau INFO ; un notebook noierait ses tableaux sous ces lignes.
setup_logging(level="WARNING")

# --- Configuration : exactement celle de `python -m src.main` -----------------------------------
# Les notebooks travaillent sur un corpus réduit (64 documents) quand `data/raw` est
# vide : l'exécution reste rapide. Si le corpus de référence a été généré par `make data`, il est
# utilisé tel quel — les valeurs absolues d'un notebook ne sont pas celles de la référence.
NB_DOCUMENTS = 64

GlobalHydra.instance().clear()
with initialize_config_dir(config_dir=str(PROJECT_ROOT / "conf"), version_base=None):
    CONFIG = validate_config(
        compose(
            config_name="config",
            overrides=[
                "mode=train",
                f"data.n_samples={NB_DOCUMENTS}",
                "seed=42",
                "log_level=WARNING",
                "train.callbacks.progress_bar=false",
            ],
        )
    )

PATHS = ProjectPaths.from_root(PROJECT_ROOT)
# Les notebooks lisent le corpus du projet et écrivent leurs artefacts dans `outputs/notebooks` :
# ils ne doivent jamais écraser ceux produits par `make train` / `make evaluate`.
NB_PATHS = ProjectPaths(
    root=PROJECT_ROOT,
    data_dir=PATHS.data_dir,
    artifacts_dir=PROJECT_ROOT / "outputs" / "notebooks" / "artifacts",
).ensure()

CHUNKING = dict(node(CONFIG, "preprocessing").get("chunking", {}))

print(f"Projet            : {CONFIG.project.name}")
print(f"Tâche             : {CONFIG.metrics.task}")
print(f"Métrique primaire : {CONFIG.metrics.primary} (plancher contractuel : 0.6)")
print(f"Algorithme        : {CONFIG.model.algorithm} ({CONFIG.model.name})")
print(
    f"Découpage         : {CHUNKING.get('max_tokens')} tokens, "
    f"chevauchement {CHUNKING.get('overlap_tokens')}"
)
print(f"Sorties notebook  : {NB_PATHS.artifacts_dir.relative_to(PROJECT_ROOT)}")

## 1. Le corpus

Un assistant documentaire ne vaut que par son corpus : ce que l'organisation a écrit, où, quand et pour qui. Les quatre lectures ci-dessous — source, section, longueur, date — suffisent à détecter les deux pathologies classiques : une source qui écrase toutes les autres, et un corpus périmé dont les réponses ne sont plus valides.

In [ ]:
from src.data.generators import SyntheticCorpusGenerator
from src.data.loaders import TextCorpusLoader

LOADER = TextCorpusLoader(PATHS, formats=CONFIG.data.formats)
if LOADER.documents_path.exists():
    DOCUMENTS = LOADER.load_documents()
    QUERIES = LOADER.load_queries()
    METADATA = LOADER.load_metadata()
    print(f"Corpus lu depuis data/raw : {len(DOCUMENTS)} documents, {len(QUERIES)} questions")
else:
    # Un clone frais n'a pas encore de données : le notebook reste exécutable. Le corpus réduit est
    # persisté pour que les pipelines (train, evaluate) trouvent les mêmes fichiers que le
    # notebook ; `make data` régénère le corpus de référence.
    bundle = SyntheticCorpusGenerator.from_config(CONFIG.data).generate()
    DOCUMENTS, QUERIES, METADATA = bundle.documents, bundle.queries, bundle.metadata
    LOADER.save_documents(DOCUMENTS)
    LOADER.save_queries(QUERIES)
    LOADER.save_metadata(METADATA)
    print(f"data/raw absent : corpus de {len(DOCUMENTS)} documents généré et persisté")

CALIBRATION = QUERIES[QUERIES["split"] == "calibration"].reset_index(drop=True)
VALIDATION = QUERIES[QUERIES["split"] == "val"].reset_index(drop=True)
TEST = QUERIES[QUERIES["split"] == "test"].reset_index(drop=True)
ANSWERABLE = QUERIES[QUERIES["answer_type"] != "unanswerable"].reset_index(drop=True)
# Sous-ensemble utilisé pour **régler** quoi que ce soit : la validation seule. Le test ne sert
# qu'à juger, jamais à choisir (y compris dans un notebook).
VAL_ANSWERABLE = VALIDATION[VALIDATION["answer_type"] != "unanswerable"].reset_index(drop=True)
print(
    "splits :",
    {
        name: len(frame)
        for name, frame in (("calibration", CALIBRATION), ("val", VALIDATION), ("test", TEST))
    },
)

In [ ]:
DOCUMENTS.head(3)[["doc_id", "title", "section", "source", "published_at", "n_tokens"]]

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
DOCUMENTS["source"].value_counts().plot.bar(ax=axes[0], title="Documents par source", rot=30)
DOCUMENTS["section"].value_counts().head(10).plot.bar(
    ax=axes[1], title="Documents par section", rot=30
)
DOCUMENTS["n_tokens"].plot.hist(ax=axes[2], bins=30, title="Longueur des documents (tokens)")
plt.tight_layout()
plt.show()

print(
    "période couverte       :",
    DOCUMENTS["published_at"].min().date(),
    "->",
    DOCUMENTS["published_at"].max().date(),
)
print("identifiants dupliqués :", int(DOCUMENTS["doc_id"].duplicated().sum()))
print("sections distinctes    :", DOCUMENTS["section"].nunique())

**Ce qu'il faut retenir**

- Une source qui concentre la majorité des documents oriente toutes les réponses : la ventilation par source est un prérequis à la lecture du rappel.
- Les longueurs fixent le nombre de passages : un document de 40 tokens ne produit qu'un passage, un document de 400 en produit quatre.
- Des identifiants dupliqués casseraient les jointures entre passages, réponses et questions : le contrat les refuse, et cette cellule les compte.

## 2. Le jeu de questions

Le corpus dit ce qui est **écrit**, les questions disent ce qui est **demandé**. La difficulté annotée n'est pas décorative : elle indique à quel titre une question compte. Une question hors corpus ne se juge pas au rappel mais à l'abstention.

In [ ]:
summary = (
    QUERIES.assign(hors_corpus=QUERIES["answer_type"] == "unanswerable")
    .groupby("difficulty")
    .agg(
        questions=("query_id", "size"),
        documents_annotes=("n_gold_docs", "mean"),
        hors_corpus=("hors_corpus", "sum"),
    )
    .round(2)
)
summary

In [ ]:
fig, axes = plt.subplots(1, 3, figsize=(15, 4))
QUERIES["split"].value_counts().sort_index().plot.bar(
    ax=axes[0], title="Questions par split", rot=0
)
QUERIES["difficulty"].value_counts().plot.bar(ax=axes[1], title="Questions par difficulté", rot=30)
QUERIES["intent"].value_counts().plot.bar(ax=axes[2], title="Questions par intention", rot=30)
plt.tight_layout()
plt.show()

print("questions hors corpus     :", int((QUERIES["answer_type"] == "unanswerable").sum()))
print("questions multi-document  :", int((QUERIES["n_gold_docs"] > 1).sum()))
print("longueur moyenne question :", round(QUERIES["question"].str.len().mean(), 1), "caractères")

In [ ]:
# Une question répondable est annotée par les documents qui la portent **et** par l'extrait exact
# qui contient la réponse ; une question hors corpus n'a ni l'un ni l'autre.
easy = ANSWERABLE.sort_values("n_gold_docs").iloc[0]
out_of_corpus = QUERIES[QUERIES["answer_type"] == "unanswerable"].iloc[0]
display(
    pd.DataFrame([easy, out_of_corpus])[
        ["question", "reference_answer", "gold_doc_ids", "difficulty", "answer_type"]
    ]
)
print("extrait annoté de la question répondable :", str(easy["gold_span"])[:200])

**Ce qu'il faut retenir**

- Trois familles de questions cohabitent : retrouver une formulation identique (facile), comprendre sans les mêmes mots (paraphrase), agréger deux sources (multi-document). Elles n'appellent pas les mêmes correctifs.
- La part de questions hors corpus est le seul segment où « ne pas répondre » est la bonne réponse : sans elle, un assistant qui répond toujours paraît excellent.
- Le split de calibration sert à régler la décision de refus : ni la validation ni le test ne doivent y servir, sinon la mesure publiée est fausse.